# Porto Seguro Safe Driver Prediction – 02 Data Preparation

**Goal:** turn the decisions of the EDA log (notebook 01, section 11) into preprocessing pipelines, set up the
evaluation protocol used to compare all models, and check with a baseline and two default models that the
pipelines work.

The pipelines are defined in `src/preprocessing.py`, because notebooks 03 and 04 reuse them; this notebook
explains every step and inspects what the pipelines produce.

**Workflow**
1. Setup: imports and shared project code
2. Load data, recode missing values, split (identical to notebook 01)
3. Feature lists and missing values per type
4. Preparation for linear models (logistic regression)
5. Preparation for tree models (random forest, gradient boosting)
6. Evaluation protocol: 5-fold stratified cross-validation, Gini and PR-AUC
7. Dummy baseline and first models with default settings
8. Summary and next steps

## 1. Setup

In [1]:
# 1. Imports and shared project code
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn import set_config
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# shared project code lives in ../src (same loading, split and preparation for every notebook)
sys.path.insert(0, str(Path.cwd().parent))
from src.data_utils import RANDOM_STATE, load_data, split_features_target, train_test_parts, feature_groups
from src.preprocessing import (MIN_CATEGORY_COUNT, REDUNDANT_FOR_LINEAR, make_linear_preprocessor,
                               make_tree_preprocessor, categorical_columns)
from src.evaluation import CV_FOLDS, SCORING, evaluate

RESULTS_DIR = Path('../results')
set_config(display='text')  # print pipelines as text (readable also in the PDF export)

## 2. Load data, recode missing values and split

In [2]:
# 2. Load data (-1 recoded as NaN) and the stratified 80/20 split, shared with notebook 01
data = load_data()
features, target = split_features_target(data)
X_train, X_test, y_train, y_test = train_test_parts(features, target)

# check: must give the same numbers as in notebook 01
print(f"training part: {len(y_train):,} rows, claim rate {y_train.mean():.4%}")
print(f"test part:     {len(y_test):,} rows, claim rate {y_test.mean():.4%}")

training part: 476,169 rows, claim rate 3.6447%
test part:     119,043 rows, claim rate 3.6449%


From here on only the training part is used. The test part stays untouched until the final evaluation
(notebook 04).

## 3. Feature lists and missing values per type

In [3]:
# 3. Feature lists by type (from the column names, see notebook 01, section 7)
column_groups = feature_groups(X_train.columns)
missing_counts = X_train.isna().sum()

type_overview = pd.DataFrame({
    'n_features': {kind: len(cols) for kind, cols in column_groups.items()},
    'with_missing': {kind: [col for col in cols if missing_counts[col] > 0]
                     for kind, cols in column_groups.items()},
})
print(type_overview.to_string())

             n_features                                                                                                                             with_missing
binary               17                                                                                                                                       []
categorical          14  [ps_ind_02_cat, ps_ind_04_cat, ps_ind_05_cat, ps_car_01_cat, ps_car_02_cat, ps_car_03_cat, ps_car_05_cat, ps_car_07_cat, ps_car_09_cat]
numeric              26                                                                                                        [ps_reg_03, ps_car_11, ps_car_14]
calc                 20                                                                                                                                       []


Missing values occur only in categorical features and in three numeric features (`ps_reg_03`, `ps_car_11`,
`ps_car_14`); binary and `calc` features are complete. (The `calc` group overlaps with the other types: its 20
features are 6 binary and 14 numeric ones.)

## 4. Preparation for linear models
Logistic regression needs complete, numeric features on comparable scales. Each step follows from the EDA log:

| Step | Features | Reason (EDA section) |
|---|---|---|
| median imputation + "was missing" indicator | numeric | missingness is informative (6) |
| standardization | numeric | skewed features on different scales (7); needed for the L2 penalty |
| missing → own category | categorical | missingness is informative (6) |
| one-hot encoding, rare categories grouped | categorical | nominal codes have no order (7); avoids weights fitted to a few rows |
| pass through | binary | already 0/1, no missing values |
| drop `ps_ind_14` | — | exact sum of `ps_ind_10–13_bin` (9) |

All steps are inside one `ColumnTransformer`; in cross-validation it is fitted on the training folds only, so
medians, scales and category lists never see the validation fold (no leakage).

In [4]:
# 4. Build the linear preprocessor and fit it once on the training part, only to inspect its output
linear_preprocessor = make_linear_preprocessor(X_train.columns)
linear_preprocessor

ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('impute',
                                                  SimpleImputer(add_indicator=True,
                                                                strategy='median')),
                                                 ('scale', StandardScaler())]),
                                 ['ps_ind_01', 'ps_ind_03', 'ps_ind_15',
                                  'ps_reg_01', 'ps_reg_02', 'ps_reg_03',
                                  'ps_car_11', 'ps_car_12', 'ps_car_13',
                                  'ps_car_14', 'ps_car_15', 'ps_calc_01',
                                  'ps_calc_02', 'ps_calc_03', 'ps_calc_04',
                                  'ps_calc_05', '...
                                  'ps_car_10_cat', 'ps_car_11_cat']),
                                ('binary', 'passthrough',
                                 ['ps_ind_06_bin', 'ps_ind_07_bin',
                             

In [5]:
# 4. What comes out: one row per driver, one column per prepared feature
X_train_linear = linear_preprocessor.fit_transform(X_train)
linear_feature_names = linear_preprocessor.get_feature_names_out()

print(f"input:  {X_train.shape[1]} features")
print(f"output: {X_train_linear.shape[1]} features, stored as a sparse matrix "
      f"({X_train_linear.nnz / np.prod(X_train_linear.shape):.0%} non-zero values)")
print(f"dropped: {REDUNDANT_FOR_LINEAR}")

output_kinds = pd.Series(['missing indicator' if name.startswith('missingindicator_')
                          else 'one-hot' if '_cat_' in name
                          else 'binary' if name.endswith('_bin')
                          else 'numeric (standardized)' for name in linear_feature_names])
print("\noutput columns by kind:")
print(output_kinds.value_counts().to_string())

input:  57 features
output: 227 features, stored as a sparse matrix (20% non-zero values)
dropped: ['ps_ind_14']

output columns by kind:
one-hot                   182
numeric (standardized)     25
binary                     17
missing indicator           3


In [6]:
# 4. Which categories were grouped as "infrequent" (fewer than MIN_CATEGORY_COUNT rows)?
one_hot_encoder = linear_preprocessor.named_transformers_['categorical'].named_steps['one_hot']
categorical_input = linear_preprocessor.transformers_[1][2]

rows = []
for col, infrequent in zip(categorical_input, one_hot_encoder.infrequent_categories_):
    if infrequent is not None:
        n_rows = X_train[col].fillna(-1).isin(infrequent).sum()
        rows.append({'feature': col,
                     'grouped categories (-1 = missing)': ', '.join(str(int(value)) for value in infrequent),
                     'rows': n_rows})
print(f"threshold: {MIN_CATEGORY_COUNT} rows ({MIN_CATEGORY_COUNT / len(X_train):.2%} of the training part)\n")
print(pd.DataFrame(rows).to_string(index=False))

threshold: 500 rows (0.11% of the training part)

      feature grouped categories (-1 = missing)  rows
ps_ind_02_cat                                -1   165
ps_ind_04_cat                                -1    62
ps_car_01_cat                                -1    84
ps_car_02_cat                                -1     4
ps_car_04_cat                           4, 5, 7   751
ps_car_09_cat                                -1   458
ps_car_10_cat                                 2   144


In [7]:
# 4. Checks: no missing values left, and the standardized numeric features have mean 0 and std 1
dense_numeric = X_train_linear[:, :len(column_groups['numeric']) - 1].toarray()  # numeric block comes first
print("missing values in the output:", np.isnan(X_train_linear.data).sum())
print(f"numeric block: mean in [{dense_numeric.mean(axis=0).min():.3f}, {dense_numeric.mean(axis=0).max():.3f}], "
      f"std in [{dense_numeric.std(axis=0).min():.3f}, {dense_numeric.std(axis=0).max():.3f}]")

missing values in the output: 0


numeric block: mean in [-0.000, 0.000], std in [1.000, 1.000]


**Observations:**
- The 57 input features become 227 prepared features; 182 of them are one-hot columns, 104 alone from
  `ps_car_11_cat`. Storing them as a sparse matrix (20% non-zero) keeps the memory low enough to run the
  5 cross-validation folds in parallel.
- Only 3 missing indicators are created, because only `ps_reg_03`, `ps_car_11` and `ps_car_14` have missing
  numeric values in the training part.
- The grouping of rare categories hardly changes the data: no category of `ps_car_11_cat` is below 500 rows.
  For five features the only rare category is "missing" itself (e.g. 84 rows for `ps_car_01_cat`), so the
  infrequent column is exactly the missing flag and its strong signal (EDA section 6) is kept.
  Only `ps_car_04_cat` (categories 4, 5, 7) and `ps_car_10_cat` (category 2) have real categories merged.
- Checks passed: no missing values remain, and every standardized numeric feature has mean 0 and std 1.

## 5. Preparation for tree models
Decision trees, random forests and gradient boosting do not need imputation, scaling or one-hot encoding:
- missing values are handled natively (scikit-learn sends them to the better side of each split), so the
  information "was missing" is kept automatically;
- splits only depend on the order of the values, so scaling and skewness do not matter;
- redundant features (`ps_ind_14`, the one-hot blocks) do not hurt a tree.

The only preparation is the column selection (later used to drop the `calc` features). For histogram gradient
boosting the categorical features are declared as such, so that it can split on groups of categories instead
of treating the codes as ordered numbers.

In [8]:
# 5. Build the tree preprocessor and inspect its output
tree_preprocessor = make_tree_preprocessor(X_train.columns)
X_train_tree = tree_preprocessor.fit_transform(X_train)

print(f"output: {X_train_tree.shape[1]} features (unchanged), type {type(X_train_tree).__name__}")
print(f"missing values kept as NaN: {X_train_tree.isna().sum().sum():,}")
print(f"categorical features declared for gradient boosting: {len(categorical_columns(X_train.columns))}")

output: 57 features (unchanged), type DataFrame
missing values kept as NaN: 677,195
categorical features declared for gradient boosting: 14


## 6. Evaluation protocol
- **5-fold stratified cross-validation** on the training part: every fold keeps the 3.6% claim rate, every
  model is scored on the same folds, and the spread over the folds shows how stable a result is.
- **Main metric:** normalised Gini = 2·AUC − 1 (the competition metric). It only depends on the ranking of the
  predicted probabilities, so it is not distorted by the class imbalance.
- **Second metric:** PR-AUC (average precision), which focuses on the rare positive class; its value for a
  random model equals the claim rate (0.036).
- The preprocessing is part of each pipeline, so it is refitted inside every fold.
- **Train Gini** (on the training folds) is reported as well, to detect overfitting.

The protocol is defined in `src/evaluation.py`, so that notebooks 02 and 03 use exactly the same folds:
`CV_FOLDS` (the 5 stratified folds, fixed seed), `SCORING` (ROC-AUC and average precision) and `evaluate()`,
which runs the cross-validation and returns Gini, PR-AUC, train Gini and fit time.

In [9]:
# 6. Stratified 5-fold cross-validation and the metrics (imported in section 1, shared with notebook 03)
print(CV_FOLDS)
print(SCORING)

# check: every validation fold keeps the claim rate of the training part
fold_claim_rates = [y_train.iloc[validation_rows].mean() for _, validation_rows in CV_FOLDS.split(X_train, y_train)]
print("claim rate per validation fold:", [f"{rate:.4%}" for rate in fold_claim_rates])

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
{'roc_auc': 'roc_auc', 'pr_auc': 'average_precision'}


claim rate per validation fold: ['3.6447%', '3.6447%', '3.6447%', '3.6447%', '3.6447%']


## 7. Dummy baseline and first models
The dummy model predicts the overall claim rate for every driver: the reference every model has to beat.
Logistic regression and histogram gradient boosting are then run with **default** settings, one per
preprocessing pipeline, only to check that the pipelines work end to end. The systematic comparison and the
tuning follow in notebook 03.

In [10]:
# 7. Three models: dummy baseline, logistic regression (linear pipeline), gradient boosting (tree pipeline)
models = {
    'Dummy (constant claim rate)': DummyClassifier(strategy='prior'),
    'Logistic regression (default)': Pipeline([
        ('prepare', make_linear_preprocessor(X_train.columns)),
        ('model', LogisticRegression(max_iter=1000)),
    ]),
    'Hist. gradient boosting (default)': Pipeline([
        ('prepare', make_tree_preprocessor(X_train.columns)),
        ('model', HistGradientBoostingClassifier(categorical_features=categorical_columns(X_train.columns),
                                                 random_state=RANDOM_STATE)),
    ]),
}

results = [evaluate(model, X_train, y_train, name) for name, model in models.items()]
results_df = pd.DataFrame(results).set_index('model')
print(results_df.round(4).to_string())

                                   gini_mean  gini_std  pr_auc_mean  pr_auc_std  train_gini  fit_time_s
model                                                                                                  
Dummy (constant claim rate)           0.0000    0.0000       0.0364      0.0000      0.0000      0.9516
Logistic regression (default)         0.2573    0.0063       0.0637      0.0012      0.2721     16.9057
Hist. gradient boosting (default)     0.2525    0.0100       0.0618      0.0013      0.4071     33.7638


In [11]:
# 7. Save the results, so that notebook 03 can extend the comparison table
RESULTS_DIR.mkdir(exist_ok=True)
results_df.to_csv(RESULTS_DIR / '02_first_models_cv.csv')
print("saved to", RESULTS_DIR / '02_first_models_cv.csv')

saved to ..\results\02_first_models_cv.csv


**Observations:**
- Both pipelines work end to end and both models are far above the dummy baseline: CV Gini 0.257
  (logistic regression) and 0.253 (gradient boosting) against 0; PR-AUC 0.064 and 0.062 against 0.036.
- The fold-to-fold spread (std 0.006–0.010) is about as large as the difference between the two models, so
  with default settings neither is clearly better.
- Logistic regression barely overfits (train Gini 0.272 vs CV 0.257): it is limited by its linear form,
  not by the amount of data.
- Gradient boosting overfits strongly (train Gini 0.407 vs CV 0.253): with default settings its trees adapt
  to noise. Tuning (smaller learning rate, stronger regularization, fewer leaves) is expected to help most here.
- For reference, the best Kaggle solutions reached a Gini of about 0.29 on the competition test set.

## 8. Summary and next steps

- **Linear pipeline:** median + missing indicator and standardization for numeric features, missing as own
  category and one-hot encoding (rare categories grouped) for categorical features, binary passed through,
  `ps_ind_14` dropped: 57 → 227 features, no missing values.
- **Tree pipeline:** column selection only; NaN kept, categorical features declared for gradient boosting.
- **Protocol:** stratified 5-fold CV on the training part with Gini (main) and PR-AUC; preparation refitted
  inside each fold, so there is no leakage.
- **First results:** logistic regression 0.257 and gradient boosting 0.253 CV Gini (dummy: 0); boosting
  overfits with default settings and needs tuning.

**Next steps (notebook 03):**
1. `calc` ablation: CV Gini with and without the 20 `calc` features.
2. More model families from the course: decision tree, Naive Bayes, random forest, MLP.
3. Class imbalance: compare `class_weight='balanced'` with no weighting.
4. Hyperparameter tuning of the best one or two models (randomized search on the same CV folds).
5. Feature engineering ideas: number of missing values per row; interactions of the strongest features.